In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
from PIL import Image
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy import ndimage as ndi
from tqdm.notebook import tqdm

In [ ]:
# prepare data
noisy_labels_dir = Path('../data/external/dataset/noisy_labels')
fp_noisy_labels = sorted(list(noisy_labels_dir.rglob('*.png')))
clean_labels_dir = Path('../data/external/dataset/clean_labels')
fp_clean_labels = sorted(list(clean_labels_dir.rglob('*.png')))
imgs_dir = Path('../data/external/dataset/image_patches/')
fp_imgs = sorted(list(imgs_dir.rglob('*.png')))
assert len(fp_noisy_labels) == len(fp_clean_labels) == len(fp_imgs) == 5000

# map the filenames to the filepaths
fp_imgs = {fp.stem: fp for fp in fp_imgs}
fp_noisy_labels = {fp.stem: fp for fp in fp_noisy_labels}
fp_clean_labels = {fp.stem: fp for fp in fp_clean_labels}

In [ ]:
def compute_image_level_metrics(img_pred, img_true):
    """
    Compute IoU, Precision, Recall, Accuracy and F1 score between predicted mask and ground truth mask.
    """

    # make sure pred and mask are boolean arrays
    assert img_pred.dtype == bool, "Predicted mask should be a boolean array."
    assert img_true.dtype == bool, "Ground truth mask should be a boolean array."

    tp = np.sum(img_true & img_pred)
    tn = np.sum(~img_true & ~img_pred)
    fp = np.sum(~img_true & img_pred)
    fn = np.sum(img_true & ~img_pred)

    iou = tp / max(tp + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    f1_score = 2 * tp / max(2 * tp + fp + fn, 1)

    stats = {
        'iou': iou,
        'precision': precision,
        'recall': recall,
        'accuracy': accuracy,
        'f1_score': f1_score
    }

    return stats

In [ ]:
def compute_instance_level_iou(
        img_pred: np.ndarray,
        img_true: np.ndarray,
        min_n_px_per_instance: int = 0,
        connectivity: int = 2,
        export_images: bool = False
):
    """ GT-centric instance matcher

    - it labels GT/pred instances, drops tiny components in-place (from both GT and pred), then relabels.
    - it builds a pairwise intersection table, assigns each prediction to the single GT with the largest overlap (or -1 if none),
    and only groups predictions that map to the same GT into one match
    (never group predictions across different GTs and never assign one prediction to multiple gt)
    - predictions with zero overlap stay separate FPs; GT instances with no assigned predictions are FNs.

    :param img_pred: predicted binary mask
    :param img_true: ground truth binary mask
    :param min_n_px_per_instance: minimum number of pixels per instance to keep from both GT and pred
    :param connectivity: 1 for 4-connected, 2 for 8-connected
    :param export_images: if True, returns additional images for plotting
    :return: A tuple with:
        - matches: list of dictionaries, each containing:
            - gt_id: int, 1-based index of the GT instance
            - pred_ids: list of int, 1-based indices of the assigned predictions
            - gt_area: int, area of the GT instance
            - pred_areas: list of int, areas of the assigned predictions
            - intersection: int, sum of overlaps with those predictions
            - iou: float, IoU of the GT and the union of the assigned predictions
        - (for plotting) when export_images is True, a dictionary with:
            - gt_lab, pr_lab: integer label maps (0 = background)
            - gt_to_group, pr_to_group: lists mapping instance id→group id (0 if none)
            - gt_group_map, pr_group_map: per-pixel group id images (0 = background)
    """

    assert img_true.shape == img_pred.shape, "gt and pred must have same shape"
    img_true = img_true.astype(bool)
    img_pred = img_pred.astype(bool)

    # Label connected components
    struct = ndi.generate_binary_structure(img_true.ndim, connectivity)
    gt_lab, n_gt = ndi.label(img_true, structure=struct)
    pr_lab, n_pr = ndi.label(img_pred, structure=struct)

    # Drop small components in-place, then relabel
    if min_n_px_per_instance > 0:
        if gt_lab.max() > 0:
            gt_areas_tmp = np.bincount(gt_lab.ravel())
            drop = np.where(gt_areas_tmp < min_n_px_per_instance)[0]
            drop = drop[drop != 0]
            if drop.size:
                gt_lab[np.isin(gt_lab, drop)] = 0

        if pr_lab.max() > 0 and min_n_px_per_instance > 0:
            pr_areas_tmp = np.bincount(pr_lab.ravel())
            drop = np.where(pr_areas_tmp < min_n_px_per_instance)[0]
            drop = drop[drop != 0]
            if drop.size:
                pr_lab[np.isin(pr_lab, drop)] = 0

        # relabel after dropping small components
        gt_lab, n_gt = ndi.label(gt_lab > 0, structure=struct)
        pr_lab, n_pr = ndi.label(pr_lab > 0, structure=struct)

    # Empty GT with no predictions
    if n_gt == 0 and n_pr == 0:
        return [], None

    # Compute pair-wise intersections
    if n_gt > 0 and n_pr > 0:
        h = np.histogram2d(
            gt_lab.ravel(), pr_lab.ravel(),
            bins=(n_gt + 1, n_pr + 1),
            range=[[0, n_gt + 1], [0, n_pr + 1]]
        )[0][1:, 1:]
        h = h.astype(np.int64)
    else:
        h = np.zeros((n_gt, n_pr), dtype=np.int64)

    gt_areas = np.bincount(gt_lab.ravel(), minlength=n_gt + 1)[1:].astype(np.int64) \
        if n_gt else np.array([], dtype=np.int64)
    pr_areas = np.bincount(pr_lab.ravel(), minlength=n_pr + 1)[1:].astype(np.int64) \
        if n_pr else np.array([], dtype=np.int64)

    # Assign each pred to the single GT with the largest overlap; -1 if no overlap.
    if n_pr > 0 and n_gt > 0:
        best_gt_for_pred = h.argmax(axis=0)
        best_overlap = h.max(axis=0)
        assigned_gt_0based = np.where(best_overlap > 0, best_gt_for_pred, -1)
    elif n_pr > 0:
        assigned_gt_0based = -np.ones(n_pr, dtype=int)
    else:
        assigned_gt_0based = np.array([], dtype=int)

    # Start building the matches
    matches = []

    # TP groups (GTs with at least one assigned prediction)
    for gi in range(n_gt):
        pred_ids = np.where(assigned_gt_0based == gi)[0]
        if pred_ids.size > 0:
            inter = int(h[gi, pred_ids].sum()) if h.size else 0
            up = int(pr_areas[pred_ids].sum()) if pr_areas.size else 0
            ug = int(gt_areas[gi]) if gt_areas.size else 0
            denom = ug + up - inter
            group_iou = (inter / denom) if denom > 0 else 0.0
            matches.append({
                "gt_id": int(gi + 1),
                "pred_ids": [int(pid + 1) for pid in pred_ids.tolist()],
                "gt_area": ug,
                "pred_areas": [int(pr_areas[pid]) for pid in pred_ids.tolist()],
                "intersection": inter,
                "iou": float(group_iou),
            })

    # FN groups (GTs with no assigned preds)
    for gi in range(n_gt):
        if not np.any(assigned_gt_0based == gi):
            ug = int(gt_areas[gi]) if gt_areas.size else 0
            matches.append({
                "gt_id": int(gi + 1),
                "pred_ids": [],
                "gt_area": ug,
                "pred_areas": [],
                "intersection": 0,
                "iou": 0.0,
            })

    # FP groups (predictions with no assigned GT)
    for pj in range(n_pr):
        if assigned_gt_0based.size == 0 or assigned_gt_0based[pj] < 0:
            pa = int(pr_areas[pj]) if pr_areas.size else 0
            matches.append({
                "gt_id": -1,
                "pred_ids": [int(pj + 1)],
                "gt_area": 0,
                "pred_areas": [pa],
                "intersection": 0,
                "iou": 0.0,
            })

    # Assign group IDs and build ID to group maps
    for k, m in enumerate(matches, start=1):
        m["group_id"] = int(k)

    if not export_images:
        return matches, None

    gt_to_group = [0] * n_gt
    pr_to_group = [0] * n_pr
    for m in matches:
        gid = m["group_id"]
        if m["gt_id"] > 0:
            gt_to_group[m["gt_id"] - 1] = gid
        for pid in m["pred_ids"]:
            pr_to_group[pid - 1] = gid

    # Per-pixel group maps for plotting
    gt_group_map = np.zeros_like(gt_lab, dtype=int)
    pr_group_map = np.zeros_like(pr_lab, dtype=int)
    for i, gid in enumerate(gt_to_group, start=1):
        if gid:
            gt_group_map[gt_lab == i] = gid
    for j, gid in enumerate(pr_to_group, start=1):
        if gid:
            pr_group_map[pr_lab == j] = gid

    imgs_out = dict(
        gt_lab=gt_lab,
        pr_lab=pr_lab,
        gt_to_group=gt_to_group,
        pr_to_group=pr_to_group,
        gt_group_map=gt_group_map,
        pr_group_map=pr_group_map,
    )
    return matches, imgs_out

#### Ensemble the predictions (from the last epoch)

In [ ]:
pred_root_dir = Path('../data/external/experiments/jakob/ensemble/')
seeds = [42 + i for i in range(10)]
pred_dirs = [pred_root_dir / f"cv_iter_0_seed_{seed}/version_0/predictions" for seed in seeds]

# image name -> list of predictions
preds_all = {k: [] for k in fp_clean_labels}
epoch = 30  # last epoch

for pred_dir in tqdm(pred_dirs, desc="Loading predictions"):
    fp_results = pred_dir / f"saved_predictions_{epoch - 1:03d}.npz"
    assert fp_results.exists(), f"File {fp_results} does not exist."

    # Load the predictions
    results = np.load(fp_results)
    preds = results['preds'].squeeze()  # shape: (5000, 256, 256)
    names = results['names']  # shape: (5000,)

    assert len(preds) == len(names) == 5000, "Mismatch in number of predictions and names."

    # Save them to the dictionary
    for i, name in enumerate(names):
        pred = (preds[i].astype(np.float32) / 255.0)  # predictions were saved as uint8 in [0, 255]
        preds_all[name].append(pred)

In [ ]:
# Compute the ensemble prediction by averaging
# first take the average, then threshold
preds = {}
for name, preds_list in preds_all.items():
    preds[name] = np.mean(preds_list, axis=0) >= 0.5  # threshold at 0.5

#### Compute the metrics for:
1. predictions vs noisy labels
2. predictions vs clean labels (both at image level and instance level)
3. noisy labels vs clean labels

In [ ]:
stats_pred_vs_clean = []
stats_pred_vs_noisy = []
stats_noisy_vs_clean = []

# minimum number of pixels per instance to count it (either in GT or prediction)
min_n_px_per_inst = 100
min_iou_per_inst = 0.01  # minimum IoU to consider a match (used for stats only)
stats_inst = []  # w.r.t. the clean labels

for fn in tqdm(preds):
    pred = preds[fn]
    mask_noisy = np.array(Image.open(fp_noisy_labels[fn])) > 0
    mask_clean = np.array(Image.open(fp_clean_labels[fn])) > 0

    crt_stats_noisy = compute_image_level_metrics(pred, mask_noisy)
    crt_stats_noisy['img'] = fn
    stats_pred_vs_noisy.append(crt_stats_noisy)

    crt_stats_clean = compute_image_level_metrics(pred, mask_clean)
    crt_stats_clean['img'] = fn
    crt_stats_clean['area_pos_px'] = np.sum(mask_clean)
    stats_pred_vs_clean.append(crt_stats_clean)

    crt_stats_noisy_vs_clean = compute_image_level_metrics(mask_noisy, mask_clean)
    crt_stats_noisy_vs_clean['img'] = fn
    stats_noisy_vs_clean.append(crt_stats_noisy_vs_clean)

    crt_matches, _ = compute_instance_level_iou(
        pred, mask_clean, min_n_px_per_instance=min_n_px_per_inst
    )
    crt_matches_df = pd.DataFrame(crt_matches)
    crt_matches_df['img'] = fn
    stats_inst.append(crt_matches_df)

stats_df_preds_vs_noisy = pd.DataFrame(stats_pred_vs_noisy)
stats_df_preds_vs_clean = pd.DataFrame(stats_pred_vs_clean)
stats_df_noisy_vs_clean = pd.DataFrame(stats_noisy_vs_clean)
stats_df_inst = pd.concat(stats_inst, ignore_index=True)
stats_df_inst['is_matched'] = stats_df_inst['iou'] > min_iou_per_inst

print("Statistics for predictions vs Noisy Labels:")
display(stats_df_preds_vs_noisy.describe())
print("Statistics for predictions vs Clean Labels:")
display(stats_df_preds_vs_clean.describe())
print("Statistics for Noisy Labels vs Clean Labels:")
display(stats_df_noisy_vs_clean.describe())
print("Instance-level statistics for predictions vs Clean Labels:")
display(stats_df_inst.describe())

In [ ]:
def counts_from_matches(matches):
    # total unique instances
    gt_ids = {m["gt_id"] for m in matches if m["gt_id"] > 0}
    n_gt = len(gt_ids)

    pred_ids = set()
    for m in matches:
        pred_ids.update(m["pred_ids"])
    n_pred = len(pred_ids)

    # counts (group-wise tp = gt with >=1 pred; fp = unmatched preds; fn = unmatched gts)
    tp = sum(1 for m in matches if m["gt_id"] > 0 and len(m["pred_ids"]) > 0)
    fn = sum(1 for m in matches if m["gt_id"] > 0 and len(m["pred_ids"]) == 0)
    fp = sum(len(m["pred_ids"]) for m in matches if m["gt_id"] == -1)

    # sanity checks:
    assert tp + fn == n_gt
    assert fp + sum(len(m["pred_ids"]) for m in matches if m["gt_id"] > 0) == n_pred

    return dict(n_gt=n_gt, n_pred=n_pred, tp=tp, fp=fp, fn=fn)

stats_df_inst_counts = stats_df_inst.groupby('img').apply(lambda sdf: counts_from_matches(sdf.to_dict('records')), include_groups=False)
stats_df_inst_counts = pd.DataFrame(stats_df_inst_counts.tolist(), index=stats_df_inst_counts.index)
stats_df_inst_counts

#### For a single image, show the predictions, masks and stats

In [ ]:
plt.figure(dpi=120)
stats_df_inst.iou.hist(bins=50)
plt.xlabel('IoU')
plt.title(
    f"Distribution of instance IoU"
    f"\n#images = {len(set(stats_df_inst.img))}, #instances = {len(stats_df_inst)})"
    f"\nTP = {stats_df_inst_counts.tp.sum()}, "
    f"FP = {stats_df_inst_counts.fp.sum()}, "
    f"FN = {stats_df_inst_counts.fn.sum()}"
)

plt.figure(dpi=120)
_df = stats_df_inst.copy()

# Clip the very large areas
_df.gt_area = _df.gt_area.clip(upper=_df.gt_area.quantile(0.99))

_df[_df.is_matched].gt_area.hist(bins=50, label=f'GT area for IoU >= {min_iou_per_inst:.1%}')
_df[~_df.is_matched].gt_area.hist(bins=50, label=f'GT area for IoU < {min_iou_per_inst:.1%}')
plt.xlabel('GT area (px)')
# plt.xscale('log')
plt.legend()
plt.title('GT area separated by IoU > 0')

In [ ]:
# number of matched instances per image
n_inst_per_img = stats_df_inst.groupby('img').size()
n_inst_per_img_matched = stats_df_inst.groupby('img').is_matched.sum()
ratio_matched = n_inst_per_img_matched / n_inst_per_img
_df = pd.DataFrame({
    'n_inst': n_inst_per_img,
    'n_matched': n_inst_per_img_matched,
    'ratio_matched': ratio_matched
})
_df = _df.sort_values(['n_inst', 'n_matched'])

plt.figure(figsize=(20, 5), dpi=120)
x = np.arange(len(_df))
plt.bar(x, _df.n_inst, label='Number of instances', alpha=0.7)
plt.bar(x, _df.n_matched, label='Number of matched instances', alpha=0.85)
plt.grid()
plt.xlabel('Image index sorted by number of instances')
plt.ylabel('Number of instances')
plt.title(f'Number of instances and matched instances per image (#images = {len(_df)})')
plt.legend(loc='upper left')
# plt.xlim(len(n_inst_per_img) - 150, len(n_inst_per_img))  # zoom in on the images with many instances
plt.show()

# check the images with the lowest ratio but many instances
_df_low_ratio = _df[_df.n_inst > 10].sort_values('ratio_matched').head(10)
_df_low_ratio

In [ ]:
# index of the image to plot
# i_img = 0
# i_img = np.random.randint(len(preds))
i_img = 1881  # filters out two small GT objects
# i_img = 3923  # bloby prediction (one instance matching multiple GTs)

# Get some images with many instances and low detection
# i_img = list(preds.keys()).index(_df_low_ratio.index[9])

fn = list(preds.keys())[i_img]
pred = preds[fn]
img = np.array(Image.open(fp_imgs[fn]))
mask_noisy = np.array(Image.open(fp_noisy_labels[fn])) > 0
mask_clean = np.array(Image.open(fp_clean_labels[fn])) > 0

crt_stats_noisy = compute_image_level_metrics(pred, mask_noisy)
crt_stats_clean = compute_image_level_metrics(pred, mask_clean)
crt_stats_noisy_vs_clean = compute_image_level_metrics(mask_noisy, mask_clean)
crt_matches, imgs_matches = compute_instance_level_iou(
    pred, mask_clean, min_n_px_per_instance=min_n_px_per_inst, export_images=True
)
cnts = counts_from_matches(crt_matches)
crt_matches_df = pd.DataFrame(crt_matches)
display(crt_matches_df)

fig = plt.figure(figsize=(13, 11), dpi=120)

plt.subplot(2, 3, 1)
plt.imshow(mask_noisy, cmap='gray')
plt.title(
    f'Noisy Mask'
    f'\n\nIoU with clean: {crt_stats_noisy_vs_clean["iou"]:.1%}'
)

plt.subplot(2, 3, 2)
plt.imshow(mask_clean, cmap='gray')
plt.title(f'Clean Mask')

plt.subplot(2, 3, 3)
plt.imshow(pred, cmap='gray')
plt.title(
    f'Predicted Mask'
    f'\n\nIoU with clean: {crt_stats_clean["iou"]:.1%}'
    f'\nIoU with noisy: {crt_stats_noisy["iou"]:.1%}'
)

plt.subplot(2, 3, 4)
plt.imshow(img)

# ---------------------------------------------------------------------------------------------------------
# Prepare the group maps for display
# ---------------------------------------------------------------------------------------------------------
# Set non-TP (i.e. FP or FN) groups to 0 so they render as white
n_groups = max([m["group_id"] for m in crt_matches], default=0)
is_tp = np.zeros(n_groups + 1, dtype=bool)
for m in crt_matches:
    if m["gt_id"] > 0 and len(m["pred_ids"]) > 0:
        is_tp[m["group_id"]] = True

# Remap: background -> 0 (black), unmatched groups -> unmatched_idx (white), matched keep their id/colors
unmatched_idx = n_groups + 1

if n_groups > 0:
    gt_map = imgs_matches["gt_group_map"]  # per-pixel group IDs (0 = background)
    pr_map = imgs_matches["pr_group_map"]
    gt_disp = gt_map.copy()
    pr_disp = pr_map.copy()
    gt_disp[(gt_map > 0) & (~is_tp[gt_map])] = unmatched_idx
    pr_disp[(pr_map > 0) & (~is_tp[pr_map])] = unmatched_idx

    # Build colormap:
    #  - index 0 = black (background)
    #  - indices 1..n_groups = distinct colors for matched groups
    #  - index unmatched_idx = white (unmatched)
    base = np.array(plt.get_cmap('tab20').colors)  # 20 distinct rgb colors in [0,1]
    palette = np.zeros((unmatched_idx + 1, 3), float)
    palette[1:n_groups + 1] = base[np.arange(n_groups) % len(base)]
    palette[unmatched_idx] = (1.0, 1.0, 1.0)
    cmap = ListedColormap(palette)

    # Set some limits so we can show the unmatched groups in white
    bounds = np.arange(unmatched_idx + 2) - 0.5  # [-0.5, 0.5, 1.5, ..., unmatched_idx+0.5]
    norm = BoundaryNorm(bounds, cmap.N)

    plt.subplot(2, 3, 5)
    plt.imshow(gt_disp, cmap=cmap, norm=norm)
    plt.title(f"#instances: {cnts['n_gt']}")

    plt.subplot(2, 3, 6)
    plt.imshow(pr_disp, cmap=cmap, norm=norm)
    plt.title(
        f"#instances: {cnts['n_pred']}\n"
        f"TP = {cnts['tp']}, "
        f"FP = {cnts['fp']}, "
        f"FN = {cnts['fn']}\n"
        f"IoU: avg. = {crt_matches_df['iou'].mean():.1%}, "
        f"min = {crt_matches_df['iou'].min():.1%}, "
        f"max = {crt_matches_df['iou'].max():.1%}"
    )
# ---------------------------------------------------------------------------------------------------------

for i, ax in enumerate(fig.axes):
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle(f"Epoch {epoch:03d} - {fn} (i = {i_img})")
plt.tight_layout()
plt.show()

#### Compute and compare the rankings using the IoU scores

In [ ]:
noise_levels_actual = stats_df_noisy_vs_clean['iou'].values
noise_levels_pred = stats_df_preds_vs_noisy['iou'].values

# or, equivalently, use the ranks:
ranks_actual = np.zeros_like(noise_levels_actual)
ranks_actual[np.argsort(noise_levels_actual)] = np.arange(len(noise_levels_actual))
ranks_pred = np.zeros_like(noise_levels_pred)
ranks_pred[np.argsort(noise_levels_pred)] = np.arange(len(noise_levels_pred))
kendalltau = scipy.stats.kendalltau(ranks_actual, ranks_pred).statistic

plt.figure(figsize=(6, 6), dpi=120)
plt.plot(noise_levels_actual, noise_levels_pred, 'o', markersize=2)
plt.xlabel('IoU (Noisy Labels vs Clean Labels)')
plt.ylabel('IoU (Predictions vs Noisy Labels)')
plt.title(f'Kendall Tau: {kendalltau:.3f}')
plt.gca().set_aspect('equal', adjustable='box')
plt.grid()
plt.show()

In [ ]:
stats_preds_vs_clean_avg = stats_df_preds_vs_clean.mean(numeric_only=True)
print(stats_preds_vs_clean_avg.round(3))
print(f"\nkendalltau: {kendalltau:.3f}")

iou_inst_avg_all = stats_df_inst.iou.mean()
print(f"\nInstance IoU (all): {iou_inst_avg_all:.3f}")
iou_inst_avg_matched_only = stats_df_inst[stats_df_inst.is_matched].iou.mean()
print(f"Instance IoU (matched only): {iou_inst_avg_matched_only:.3f}")

vals = (
        [kendalltau]
        + [stats_preds_vs_clean_avg[k] for k in ['iou', 'accuracy', 'precision', 'recall', 'f1_score']]
        + [iou_inst_avg_all, iou_inst_avg_matched_only]
)
print()
print(' & '.join([f"{v:.3f}" for v in vals]) + ' \\\\')